# Atividade Prática 1.2 — Implementação e Governança

**Disciplina:** Processamento Digital de Imagens — BCC / UNEMAT  
**Professor:** Dr. Carlos Alex Sander J. Gulo  
**Aluno:** Lucas Tiago Siqueira  
**Semestre:** 2026/1

Este notebook resolve os 20 exercícios propostos utilizando **OpenCV**, **NumPy** e **Matplotlib**.
Todas as imagens são autorais (capturadas em celular) e carregadas diretamente do repositório público no GitHub,
de modo que o notebook executa de ponta a ponta no Google Colab sem necessidade de ajuste manual de caminhos.

## 0. Configuração do ambiente

Instala as bibliotecas necessárias (no Colab elas já vêm pré-instaladas, mas o `pip install` é idempotente)
e baixa as imagens autorais a partir do repositório GitHub usando `wget`.

In [ ]:
# Instalação das bibliotecas — funciona tanto no Colab quanto em um ambiente novo
!pip install --quiet opencv-python numpy matplotlib

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configuração padrão dos plots
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['image.cmap'] = 'gray'

print('OpenCV:', cv2.__version__)
print('NumPy :', np.__version__)

In [ ]:
# Baixa as imagens autorais diretamente do repositório público no GitHub.
# Assim o notebook roda no Colab sem precisar de upload manual.
BASE_URL = 'https://raw.githubusercontent.com/LucasTiagoUNEMAT/PDI_lucas_tiago_siqueira/main/lista_1/pics'
IMAGENS = [
    'IMG_1443.jpg',
    'IMG_1834.jpg',
    'IMG_2433.jpg',
    'IMG_2520.jpg',
    'IMG_6574.jpg',
]

os.makedirs('pics', exist_ok=True)
for nome in IMAGENS:
    destino = os.path.join('pics', nome)
    if not os.path.exists(destino):
        !wget -q -O {destino} {BASE_URL}/{nome}
    print('OK:', destino, os.path.getsize(destino), 'bytes')

In [ ]:
# Função utilitária para exibir imagens BGR (OpenCV) corretamente em RGB (Matplotlib).
def mostrar(imagem, titulo='', cmap=None):
    plt.figure()
    if imagem.ndim == 2:
        plt.imshow(imagem, cmap=cmap or 'gray')
    else:
        plt.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
    plt.title(titulo)
    plt.axis('off')
    plt.show()

# Caminhos das imagens já baixadas — usadas em todos os exercícios.
CAMINHO_IMG_PRINCIPAL = 'pics/IMG_2520.jpg'
CAMINHO_IMG_SECUNDARIA = 'pics/IMG_1834.jpg'
CAMINHO_IMG_DETALHE = 'pics/IMG_2433.jpg'

## Exercício 1 — Carregar imagem colorida e exibir com correção BGR → RGB

O OpenCV carrega imagens em **BGR**, mas o Matplotlib espera **RGB**.
Sem a conversão a foto aparece com tonalidades trocadas (azul vira vermelho).

In [ ]:
img_bgr = cv2.imread(CAMINHO_IMG_PRINCIPAL)
assert img_bgr is not None, 'Falha ao carregar a imagem principal.'

# Conversão obrigatória para exibição com Matplotlib
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure()
plt.imshow(img_rgb)
plt.title('Imagem original (RGB corrigido)')
plt.axis('off')
plt.show()

## Exercício 2 — Atributos `shape`, `size` e `dtype`

- `shape`: dimensões da matriz (altura, largura, canais).
- `size`: número total de elementos (altura × largura × canais).
- `dtype`: tipo de dado de cada pixel (normalmente `uint8`).

In [ ]:
print('shape :', img_bgr.shape)
print('size  :', img_bgr.size)
print('dtype :', img_bgr.dtype)

## Exercício 3 — Salvar cópia em formato de compressão diferente

A imagem original é JPG (compressão com perdas). Aqui salvamos uma cópia em **PNG**
(compressão sem perdas) usando `cv2.imwrite`. A extensão do arquivo determina o codec.

In [ ]:
saida_png = 'IMG_1443_copia.png'
ok = cv2.imwrite(saida_png, img_bgr)
print('Gravado com sucesso?', ok)
print('Arquivo gerado     :', saida_png, '-', os.path.getsize(saida_png), 'bytes')
print('Arquivo original   :', CAMINHO_IMG_PRINCIPAL, '-', os.path.getsize(CAMINHO_IMG_PRINCIPAL), 'bytes')

## Exercício 4 — Pixel central da imagem

Acessa o pixel localizado no centro da imagem e imprime os componentes R, G e B.
Lembrando que o OpenCV devolve o pixel na ordem (B, G, R).

In [ ]:
altura, largura = img_bgr.shape[:2]
cy, cx = altura // 2, largura // 2
b, g, r = img_bgr[cy, cx]
print(f'Pixel central em (linha={cy}, coluna={cx})')
print(f'  R = {r}')
print(f'  G = {g}')
print(f'  B = {b}')

## Exercício 5 — Pintar uma região 100×100 no canto superior

Modifica os pixels do canto superior esquerdo para uma cor sólida (vermelho).
Trabalhamos sobre uma cópia para preservar a imagem original.

In [ ]:
img_modificada = img_bgr.copy()
img_modificada[0:100, 0:100] = (0, 0, 255)  # vermelho em BGR
mostrar(img_modificada, 'Região 100x100 pintada de vermelho')

## Exercício 6 — Recorte (ROI) de um detalhe

Extrai uma sub-região (Region Of Interest) da imagem usando *fatiamento* NumPy
no formato `imagem[y1:y2, x1:x2]`.

In [ ]:
y1 = altura // 4
y2 = y1 + altura // 2
x1 = largura // 4
x2 = x1 + largura // 2
roi = img_bgr[y1:y2, x1:x2]
print('Shape do recorte:', roi.shape)
mostrar(roi, 'Recorte (ROI) central')

## Exercício 7 — Separar os canais B, G e R

`cv2.split` devolve os três canais como matrizes 2D independentes.

In [ ]:
B, G, R = cv2.split(img_bgr)
print('Canal B :', B.shape, B.dtype)
print('Canal G :', G.shape, G.dtype)
print('Canal R :', R.shape, R.dtype)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, canal, nome in zip(axes, (B, G, R), ('Azul (B)', 'Verde (G)', 'Vermelho (R)')):
    ax.imshow(canal, cmap='gray')
    ax.set_title(nome)
    ax.axis('off')
plt.show()

## Exercício 8 — Canal Verde isolado em escala de cinzentos

Como cada canal é uma matriz 2D, basta exibi-lo com colormap `gray` para visualizar
a intensidade do verde como se fosse uma imagem em tons de cinza.

In [ ]:
plt.figure()
plt.imshow(G, cmap='gray')
plt.title('Canal Verde (G) em tons de cinza')
plt.axis('off')
plt.show()

## Exercício 9 — `merge` trocando R com B

Reagrupamos os canais com a posição do vermelho e do azul invertidas.
O resultado é uma imagem com aparência "alienígena".

In [ ]:
img_trocada = cv2.merge([R, G, B])  # antes era (B, G, R)
mostrar(img_trocada, 'Canais R e B trocados')

## Exercício 10 — Conversão para tons de cinzento

Usamos `cv2.cvtColor` com a flag `COLOR_BGR2GRAY` para gerar uma imagem em escala de cinza.
Internamente o OpenCV calcula `0.114·B + 0.587·G + 0.299·R`.

In [ ]:
img_cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
print('Shape original:', img_bgr.shape, '-> shape em cinza:', img_cinza.shape)
plt.figure()
plt.imshow(img_cinza, cmap='gray')
plt.title('Imagem em tons de cinzento')
plt.axis('off')
plt.show()

## Exercício 11 — Espaço HSV: canal de Saturação

No espaço **HSV** (Hue, Saturation, Value), o canal **S** indica a "pureza" da cor.
Regiões coloridas e vivas terão S alto; áreas pálidas/cinzas terão S baixo.

In [ ]:
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
canal_s = img_hsv[:, :, 1]
plt.figure()
plt.imshow(canal_s, cmap='gray')
plt.title('Canal de Saturação (S) do HSV')
plt.axis('off')
plt.show()

## Exercício 12 — Aumentar brilho em +50 com `cv2.add`

Soma 50 a todos os pixels usando **operações saturadas** do OpenCV: valores acima de 255
ficam em 255 (não há *overflow*), diferentemente da aritmética NumPy padrão.

In [ ]:
matriz_brilho = np.ones(img_bgr.shape, dtype='uint8') * 50
img_brilho = cv2.add(img_bgr, matriz_brilho)
mostrar(img_brilho, 'Imagem com brilho aumentado em +50')

## Exercício 13 — Mesclagem de duas imagens com `addWeighted`

Carregamos duas imagens, redimensionamos para a mesma resolução e fazemos a combinação
linear `0.5·A + 0.5·B`.

In [ ]:
img_a = cv2.imread(CAMINHO_IMG_PRINCIPAL)
img_b = cv2.imread(CAMINHO_IMG_SECUNDARIA)

tamanho = (600, 400)  # (largura, altura)
img_a_r = cv2.resize(img_a, tamanho)
img_b_r = cv2.resize(img_b, tamanho)

img_mesclada = cv2.addWeighted(img_a_r, 0.5, img_b_r, 0.5, 0)
mostrar(img_mesclada, 'Mesclagem 0.5 / 0.5 com addWeighted')

## Exercício 14 — Negativo fotográfico (escala de cinza)

O negativo é obtido por `255 - pixel`: pixels claros viram escuros e vice-versa.

In [ ]:
img_negativo = 255 - img_cinza
plt.figure()
plt.imshow(img_negativo, cmap='gray')
plt.title('Negativo da imagem em cinza')
plt.axis('off')
plt.show()

## Exercício 15 — Máscara binária circular

Criamos uma máscara preta com um círculo branco no centro e usamos `bitwise_and`
para preservar apenas os pixels da imagem que ficam *dentro* do círculo.

In [ ]:
mascara = np.zeros(img_bgr.shape[:2], dtype='uint8')
raio = min(altura, largura) // 4
cv2.circle(mascara, (largura // 2, altura // 2), raio, 255, -1)

img_mascarada = cv2.bitwise_and(img_bgr, img_bgr, mask=mascara)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(mascara, cmap='gray')
axes[0].set_title('Máscara circular')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_mascarada, cv2.COLOR_BGR2RGB))
axes[1].set_title('Imagem mascarada')
axes[1].axis('off')
plt.show()

## Exercício 16 — Redução para 30% do tamanho original

`cv2.resize` com `fx=0.3` e `fy=0.3` reduz proporcionalmente largura e altura.

In [ ]:
img_reduzida = cv2.resize(img_bgr, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_AREA)
print('Shape original :', img_bgr.shape)
print('Shape reduzido :', img_reduzida.shape)
mostrar(img_reduzida, 'Imagem reduzida para 30%')

## Exercício 17 — Valores do histograma

Calcula o histograma da imagem em tons de cinza com `cv2.calcHist` e imprime
a contagem de pixels para cada um dos 256 níveis de intensidade.

In [ ]:
hist = cv2.calcHist([img_cinza], [0], None, [256], [0, 256]).flatten().astype(int)
print('Total de níveis:', len(hist))
print('Soma das contagens:', hist.sum(), '(deve bater com', img_cinza.size, ')')
for i in range(0, 256, 16):
    linha = ' '.join(f'{v:6d}' for v in hist[i:i + 16])
    print(f'[{i:3d}-{i + 15:3d}] {linha}')

## Exercício 18 — Histograma dos três canais no mesmo gráfico

Plotamos as três curvas (B, G, R) sobrepostas para comparar a distribuição
de intensidades de cada cor.

In [ ]:
cores = ('b', 'g', 'r')
rotulos = ('Azul', 'Verde', 'Vermelho')
plt.figure()
for i, (cor, rotulo) in enumerate(zip(cores, rotulos)):
    h_canal = cv2.calcHist([img_bgr], [i], None, [256], [0, 256])
    plt.plot(h_canal, color=cor, label=rotulo)
plt.title('Histograma dos canais BGR')
plt.xlabel('Intensidade')
plt.ylabel('Número de pixels')
plt.xlim([0, 256])
plt.legend()
plt.show()

## Exercício 19 — Equalização de histograma

Para demonstrar o efeito da equalização sobre uma imagem **escura**, primeiro reduzimos
artificialmente o brilho (multiplicando por 0.4) e depois aplicamos `cv2.equalizeHist`.
A equalização redistribui as intensidades, melhorando o contraste.

In [ ]:
img_escura = (img_cinza * 0.4).astype('uint8')
img_equalizada = cv2.equalizeHist(img_escura)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].imshow(img_escura, cmap='gray')
axes[0, 0].set_title('Imagem escura (entrada)')
axes[0, 0].axis('off')
axes[0, 1].imshow(img_equalizada, cmap='gray')
axes[0, 1].set_title('Após equalização de histograma')
axes[0, 1].axis('off')
axes[1, 0].hist(img_escura.ravel(), bins=256, range=(0, 256), color='gray')
axes[1, 0].set_title('Histograma — entrada')
axes[1, 1].hist(img_equalizada.ravel(), bins=256, range=(0, 256), color='gray')
axes[1, 1].set_title('Histograma — equalizada')
plt.tight_layout()
plt.show()

## Exercício 20 — Limiar (Threshold) binário simples

Aplica um limiar fixo: pixels com intensidade ≥ 127 viram 255 (branco) e os demais 0 (preto).
É a forma mais simples de separar objeto e fundo quando há bom contraste.

In [ ]:
limiar, img_binaria = cv2.threshold(img_cinza, 127, 255, cv2.THRESH_BINARY)
print('Limiar utilizado:', limiar)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_cinza, cmap='gray')
axes[0].set_title('Imagem em cinza')
axes[0].axis('off')
axes[1].imshow(img_binaria, cmap='gray')
axes[1].set_title('Threshold binário (127)')
axes[1].axis('off')
plt.show()

## Conclusão

Os 20 exercícios cobriram operações fundamentais de Processamento Digital de Imagens:
leitura/escrita, manipulação de pixels, fatiamento, separação e fusão de canais, conversões
entre espaços de cor (RGB, escala de cinza, HSV), aritmética de imagens, máscaras, redimensionamento,
histogramas, equalização e limiarização. Todos os passos foram aplicados em fotografias autorais
carregadas diretamente do repositório público no GitHub.